In [ ]:
import pandas as pd 

df = pd.read_csv('marketing_campaign.csv',sep='\t')

In [5]:
quan_col = df.select_dtypes('object')
quan_col.columns

Index(['Education', 'Marital_Status', 'Dt_Customer'], dtype='object')

Are certain education levels associated with specific marital statuses?

Which categories drive the association?

H0: Education level and marital status are independent (no association).
H1: Education level and marital status are not independent (there is an association).
	​


In [ ]:
df_ca = df[['Education', 'Marital_Status']].dropna()

In [ ]:
# Create contingency table
ct = pd.crosstab(df_ca['Education'], df_ca['Marital_Status'])
ct


Marital_Status,Absurd,Alone,Divorced,Married,Single,Together,Widow,YOLO
Education,,,,,,,,
2n Cycle,0,0,23,81,37,57,5,0
Basic,0,0,1,20,18,14,1,0
Graduation,1,1,119,433,252,286,35,0
Master,1,1,37,138,75,106,12,0
PhD,0,1,52,192,98,117,24,2


In [ ]:
# Chi-square test + (optional) CA for Education vs Marital_Status

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

# -----------------------------
# 0) Data (assumes df already exists)
# df = pd.read_csv('marketing_campaign.csv',sep='\t')
# -----------------------------
df_ca = df[["Education", "Marital_Status"]].dropna()

# -----------------------------
# 1) Hypotheses (printed for report)
# -----------------------------
print("H0: Education level and marital status are independent (no association).")
print("H1: Education level and marital status are not independent (there is an association).")

# -----------------------------
# 2) Contingency table
# -----------------------------
ct = pd.crosstab(df_ca["Education"], df_ca["Marital_Status"])
print("\nContingency table (Observed counts):")
print(ct)

# -----------------------------
# 3) Chi-square test
# -----------------------------
chi2, p_value, dof, expected = chi2_contingency(ct.values)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print("\nChi-square test results:")
print(f"  chi2 statistic = {chi2:.4f}")
print(f"  dof            = {dof}")
print(f"  p-value        = {p_value:.6g}")

alpha = 0.05
decision = "REJECT H0 (associated)" if p_value <= alpha else "FAIL TO REJECT H0 (independent)"
print(f"  Decision @ alpha={alpha}: {decision}")

# Assumptions check (expected counts)
min_expected = expected_df.to_numpy().min()
pct_ge_5 = (expected_df.to_numpy() >= 5).mean() * 100
print("\nAssumptions check:")
print(f"  Min expected count: {min_expected:.2f}")
print(f"  % cells with expected >= 5: {pct_ge_5:.1f}%")

# -----------------------------
# 4) (Optional) Correspondence Analysis (CA) + symmetric biplot
#    Run CA only if association is significant (p <= 0.05)
# -----------------------------
if p_value <= alpha:
    N = ct.to_numpy(dtype=float)
    n = N.sum()

    P = N / n
    r = P.sum(axis=1, keepdims=True)
    c = P.sum(axis=0, keepdims=True)

    Dr_inv_sqrt = np.diag(1.0 / np.sqrt(r.ravel()))
    Dc_inv_sqrt = np.diag(1.0 / np.sqrt(c.ravel()))

    S = Dr_inv_sqrt @ (P - r @ c) @ Dc_inv_sqrt
    U, s, Vt = np.linalg.svd(S, full_matrices=False)

    eig = s**2
    explained = eig / eig.sum()

    k = 2
    Sigma_half = np.diag(np.sqrt(s[:k]))

    # symmetric map
    F = Dr_inv_sqrt @ U[:, :k] @ Sigma_half  # rows (Education)
    G = Dc_inv_sqrt @ Vt.T[:, :k] @ Sigma_half  # cols (Marital_Status)

    # Plot symmetric biplot
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.scatter(F[:, 0], F[:, 1], marker="o", label="Rows: Education")
    for (x, y), lab in zip(F, ct.index.astype(str)):
        ax.text(x, y, f"  {lab}", va="center")

    ax.scatter(G[:, 0], G[:, 1], marker="^", label="Cols: Marital_Status")
    for (x, y), lab in zip(G, ct.columns.astype(str)):
        ax.text(x, y, f"  {lab}", va="center")

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel(f"Dim 1 ({explained[0]*100:.1f}%)")
    ax.set_ylabel(f"Dim 2 ({explained[1]*100:.1f}%)")
    ax.set_title("Correspondence Analysis (Symmetric Biplot): Education vs Marital_Status")
    ax.legend()
    ax.set_aspect("equal", adjustable="datalim")
    plt.tight_layout()
    plt.show()

    # Distance tables (DataFrames)
    row_distance_table = pd.DataFrame({
        "Education": ct.index,
        "Dim1": F[:, 0],
        "Dim2": F[:, 1],
        "Distance_from_origin": np.sqrt(F[:, 0]**2 + F[:, 1]**2)
    }).set_index("Education").sort_values("Distance_from_origin", ascending=False)

    col_distance_table = pd.DataFrame({
        "Marital_Status": ct.columns,
        "Dim1": G[:, 0],
        "Dim2": G[:, 1],
        "Distance_from_origin": np.sqrt(G[:, 0]**2 + G[:, 1]**2)
    }).set_index("Marital_Status").sort_values("Distance_from_origin", ascending=False)

    print("\nDistance from origin – ROWS (Education):")
    print(row_distance_table.round(4))

    print("\nDistance from origin – COLUMNS (Marital_Status):")
    print(col_distance_table.round(4))

else:
    print("\nCA skipped because chi-square test is not significant (p > 0.05).")


H0: Education level and marital status are independent (no association).
H1: Education level and marital status are not independent (there is an association).

Contingency table (Observed counts):
Marital_Status  Absurd  Alone  Divorced  Married  Single  Together  Widow  \
Education                                                                   
2n Cycle             0      0        23       81      37        57      5   
Basic                0      0         1       20      18        14      1   
Graduation           1      1       119      433     252       286     35   
Master               1      1        37      138      75       106     12   
PhD                  0      1        52      192      98       117     24   

Marital_Status  YOLO  
Education             
2n Cycle           0  
Basic              0  
Graduation         0  
Master             0  
PhD                2  

Chi-square test results:
  chi2 statistic = 27.2881
  dof            = 28
  p-value        = 0.502604

In [10]:
df

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,10870,1967,Graduation,Married,61223.0,0,1,13-06-2013,46,709,...,5,0,0,0,0,0,0,3,11,0
2236,4001,1946,PhD,Together,64014.0,2,1,10-06-2014,56,406,...,7,0,0,0,1,0,0,3,11,0
2237,7270,1981,Graduation,Divorced,56981.0,0,0,25-01-2014,91,908,...,6,0,1,0,0,0,0,3,11,0
2238,8235,1956,Master,Together,69245.0,0,1,24-01-2014,8,428,...,3,0,0,0,0,0,0,3,11,0


## CA2
- Ho: Education level and number of children are independent

In [14]:
# Create total number of children
df["kids"] = df["Kidhome"] + df["Teenhome"]

# Decode into labels
kids_map = {
    0: "No Children",
    1: "One Child",
    2: "Two Children",
    3: "Three Children"
}

df["kids_cat"] = df["kids"].map(kids_map)


In [15]:
from scipy.stats import chi2_contingency

df_ca = df[["Education", "kids_cat"]].dropna()

ct = pd.crosstab(df_ca["Education"], df_ca["kids_cat"])
ct


kids_cat,No Children,One Child,Three Children,Two Children
Education,,,,
2n Cycle,64,101,3,35
Basic,17,35,0,2
Graduation,321,579,25,202
Master,102,178,8,82
PhD,134,235,17,100


In [16]:
chi2, p, dof, expected = chi2_contingency(ct)

print(f"Chi-square: {chi2:.4f}")
print(f"p-value: {p:.6g}")

alpha = 0.05
print("Decision:", "Reject H0" if p <= alpha else "Fail to reject H0")


Chi-square: 19.3954
p-value: 0.0794219
Decision: Fail to reject H0


In [20]:
import plotly.express as px

# --------------------------------
# Prepare data
# --------------------------------
df_plot = df[["Education", "Kidhome", "Teenhome"]].dropna()

# Total children
df_plot["kids"] = df_plot["Kidhome"] + df_plot["Teenhome"]

# Decode kids into readable categories
kids_map = {
    0: "No Children",
    1: "One Child",
    2: "Two Children",
    3: "Three Children"
}

df_plot["kids_cat"] = df_plot["kids"].map(kids_map)

# --------------------------------
# Frequency + percentage
# --------------------------------
df_freq = (
    df_plot
    .groupby(["Education", "kids_cat"])
    .size()
    .reset_index(name="Freq")
)

df_freq["percent"] = (
    df_freq
    .groupby("Education")["Freq"]
    .apply(lambda x: x / x.sum() * 100)
    .reset_index(level=0, drop=True)
)

# --------------------------------
# Plot
# --------------------------------
fig = px.bar(
    df_freq,
    x="Education",
    y="percent",
    color="kids_cat",
    barmode="stack",
    text=df_freq["percent"].round(2).astype(str) + "%",
)

fig.update_layout(
    width=520,
    height=430,
    title="Stacked Bar Plot of Education vs Number of Children",
    xaxis_title="Education Level",
    yaxis_title="Percentage (%)",
    legend_title="Number of Children",
)

fig.update_traces(textposition="inside")

fig.show()


In [31]:
import plotly.graph_objects as go

fig = go.Figure()

# -----------------------------
# Rows: Education
# -----------------------------
fig.add_trace(go.Scatter(
    x=F[:, 0],
    y=F[:, 1],
    mode="markers+text",
    name="Education",
    text=ct.index.astype(str),
    textposition="top center",
    marker=dict(size=10)
))

# -----------------------------
# Columns: Number of Children
# -----------------------------
fig.add_trace(go.Scatter(
    x=G[:, 0],
    y=G[:, 1],
    mode="markers+text",
    name="Number of Children",
    text=ct.columns.astype(str),
    textposition="top center",
    marker=dict(size=12, symbol="triangle-up")
))

# -----------------------------
# Axes lines
# -----------------------------
fig.add_shape(type="line", x0=0, y0=min(F[:,1].min(), G[:,1].min()),
              x1=0, y1=max(F[:,1].max(), G[:,1].max()),
              line=dict(dash="dash"))

fig.add_shape(type="line", x0=min(F[:,0].min(), G[:,0].min()), y0=0,
              x1=max(F[:,0].max(), G[:,0].max()), y1=0,
              line=dict(dash="dash"))

# -----------------------------
# Layout
# -----------------------------
fig.update_layout(
    width=800,
    height=550,
    title="Correspondence Analysis: Education vs Number of Children",
    xaxis_title="Dim 1",
    yaxis_title="Dim 2",
    legend_title="Category",
    showlegend=True
)

fig.show()


The biplot shows a clear association between education level and number of children. Categories that are close to each other tend to occur together, while those far from the origin represent distinct family-size patterns.

Main associations 

- Basic education ↔ Three Children

    Both are far from the origin and located in opposite directions of higher education levels

    Indicates that individuals with basic education are more likely to have larger families

- Master ↔ Two Children

    Positioned close together on the right side

    Suggests that Master’s degree holders commonly have two children

- PhD ↔ One Child

    PhD is closer to One Child and below the origin

    Indicates a tendency toward smaller family size among PhD holders

- Graduation & No Children

    Both lie close to the origin

    Represent average behavior, with no strong association to a specific family size

- 2nd Cycle ↔ No Children

    Slight proximity suggests that mid-level education is linked to having no children